# 1.4 - Filtering Empty Data and Outliers
Filter NaN values to clean model's imput and remove outilers that could skew the model.

## 1: Dependencies and Imports

In [1]:
# Standard library
import os

# Third-party
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

## 2: Load Data

In [2]:
# Path to the integrated master dataset
data_path = '../data/Engineered.pkl'

if os.path.exists(data_path):
    # Load the integrated dataframe
    df = pd.read_pickle(data_path)
else:
    print(f"Error: File not found at {data_path}")

df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   order_id                        119143 non-null  object 
 1   is_delayed                      115722 non-null  float64
 2   actual_delivery_days            115722 non-null  float64
 3   estimated_delivery_margin_days  119143 non-null  int64  
 4   purchase_month                  119143 non-null  int32  
 5   purchase_day_of_week            119143 non-null  int32  
 6   product_weight_g                118290 non-null  float64
 7   product_volume_cm3              118290 non-null  float64
 8   product_category_name_english   116576 non-null  object 
 9   customer_zip_code_prefix        119143 non-null  int64  
 10  seller_zip_code_prefix          118310 non-null  float64
 11  is_same_state                   119143 non-null  int64  
 12  customer_state_n

,order_id,is_delayed,actual_delivery_days,estimated_delivery_margin_days,purchase_month,purchase_day_of_week,product_weight_g,product_volume_cm3,product_category_name_english,customer_zip_code_prefix,seller_zip_code_prefix,is_same_state,customer_state_num_pred,seller_state_num_pred,freight_value,price
0,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,15,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
1,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,15,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
2,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,15,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
3,53cdb2fc8bc7dce0b6741e2150273451,0.0,13.0,19,7,1,400.0,4693.0,perfumery,47813,31570.0,0,4,25,22.76,118.70
4,47770eb9100c2d0c44946d9cf07ec65d,0.0,9.0,26,8,2,420.0,9576.0,auto,75265,14840.0,0,8,25,19.22,159.90


## 3: Filter Data

In [3]:
# Create a deep copy to preserve the original modeling dataset
clean_df = df.copy()

print(f"Records before cleaning: {clean_df.shape[0]}")

Records before cleaning: 119143


In [4]:
# Target Validation: Remove rows where the target variable is unknown
clean_df.dropna(subset=['is_delayed'], inplace=True)
clean_df.dropna(subset=['actual_delivery_days'], inplace=True)

clean_df['is_delayed'] = clean_df['is_delayed'].astype('int64')
#clean_df['actual_delivery_days'] = clean_df['actual_delivery_days'].astype('int64')

print(f"Records remaining after removing null targets: {clean_df.shape[0]}")

Records remaining after removing null targets: 115722


Review if numerical imputation is prefered than removing those rows

In [5]:

# Numerical Imputation: Handle missing values using the median
# The median is preferred over the mean as it is more robust to outliers
numeric_features = ['product_weight_g', 'product_volume_cm3', 'freight_value', 'price']

for feature in numeric_features:
    if feature in clean_df.columns:
        median_value = clean_df[feature].median()
        clean_df[feature] = clean_df[feature].fillna(median_value)

In [6]:
# Categorical Imputation: Fill missing text labels
if 'product_category_name_english' in clean_df.columns:
    clean_df['product_category_name_english'] = clean_df['product_category_name_english'].fillna('Unknown')

In [7]:
# Residual Cleaning: Remove any remaining rows with null values in secondary columns
clean_df.dropna(inplace=True)

In [8]:
# Check cleaning completion and verify no nulls remain
print(f"Data cleaning complete. Final dataset dimensions: {clean_df.shape[0]} records.")
print("-" * 40)
print("Null Value Verification (Expected result is all zeros):")
print(clean_df.isnull().sum())

Data cleaning complete. Final dataset dimensions: 115722 records.
----------------------------------------
Null Value Verification (Expected result is all zeros):
order_id                          0
is_delayed                        0
actual_delivery_days              0
estimated_delivery_margin_days    0
purchase_month                    0
purchase_day_of_week              0
product_weight_g                  0
product_volume_cm3                0
product_category_name_english     0
customer_zip_code_prefix          0
seller_zip_code_prefix            0
is_same_state                     0
customer_state_num_pred           0
seller_state_num_pred             0
freight_value                     0
price                             0
dtype: int64


## 4: Remove Outliers: Multivariate Outlier Detection (K-Means Clustering)

In [9]:
# Select continuous physical and economic features
# Exclude identifiers, categorical variables, and targets
outlier_features = ['product_weight_g', 'product_volume_cm3', 'freight_value', 'price']

In [10]:
# Scale the data --> Standardization is essential; otherwise, features with higher magnitudes 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clean_df[outlier_features])

In [11]:
# Train K-Means: using k=5
kmeans_model = KMeans(n_clusters=5, random_state=42, n_init='auto')
clusters = kmeans_model.fit_predict(X_scaled)

In [12]:
# Measure distances to centroids --> The transform() method provides distances to all centroids
centroid_distances = kmeans_model.transform(X_scaled)
clean_df['centroid_distance'] = np.min(centroid_distances, axis=1)

In [13]:
# Define a "Soft" Threshold (99th Percentile) --> remove only the most extreme 1% of records to preserve as much data as possible
distance_threshold = np.percentile(clean_df['centroid_distance'], 99)
print(f"Distance threshold (99th percentile): {distance_threshold:.2f}")

Distance threshold (99th percentile): 4.03


In [14]:
# Execute filtering --> Only retain records falling within the threshold
outliers_count = clean_df[clean_df['centroid_distance'] > distance_threshold].shape[0]
filtered_df = clean_df[clean_df['centroid_distance'] <= distance_threshold].copy()

# Remove temporary calculation columns
filtered_df.drop(columns=['centroid_distance'], inplace=True)

In [15]:
# Print summary of results
print(f"Outliers removed: {outliers_count} extreme records.")
print(f"Final dataset dimensions after K-Means filtering: {filtered_df.shape[0]} records.")

Outliers removed: 1155 extreme records.
Final dataset dimensions after K-Means filtering: 114567 records.


In [16]:
# Display samples of the removed outliers for validation
display(clean_df[clean_df['centroid_distance'] > distance_threshold][outlier_features].head())

,product_weight_g,product_volume_cm3,freight_value,price
20,20850.0,125000.0,77.45,1299.00
362,544.0,5148.0,129.84,217.85
365,544.0,5148.0,129.84,217.85
393,18450.0,59976.0,107.43,1148.00
411,550.0,2816.0,32.09,1999.00


## 5: Save Data

In [17]:
# Ensure the data directory exists for persistence
os.makedirs('../data', exist_ok=True)

In [18]:
# Define export paths for the final cleaned and filtered dataset
csv_path = '../data/Filtered.csv'
pickle_path = '../data/Filtered.pkl'

In [19]:
# Export as CSV (Standard, human-readable format for tabular data)
filtered_df.to_csv(csv_path, index=False)
print(f"Master dataset saved in CSV format: {csv_path}")

Master dataset saved in CSV format: ../data/Filtered.csv


In [20]:
# Export as Pickle (Optimized format to preserve dtypes like int and datetime)
filtered_df.to_pickle(pickle_path)
print(f"Master dataset saved in Pickle format: {pickle_path}")

Master dataset saved in Pickle format: ../data/Filtered.pkl


In [21]:
# File size verification
csv_size_mb = os.path.getsize(csv_path) / (1024 * 1024)
print(f"CSV file size: {csv_size_mb:.2f} MB")

CSV file size: 11.68 MB
